In [1]:
import numpy as np
import pandas as pd

In [2]:
english_sentences = [
    "hello",
    "how are you",
    "i am fine",
    "good morning",
    "thank you",
    "i love you",
    "what is your name",
    "my name is john"
]

hindi_sentences = [
    "नमस्ते",
    "आप कैसे हैं",
    "मैं ठीक हूँ",
    "सुप्रभात",
    "धन्यवाद",
    "मैं तुमसे प्यार करता हूँ",
    "आपका नाम क्या है",
    "मेरा नाम जॉन है"
]

In [3]:
hindi_sentences = ["<start> " + s + " <end>" for s in hindi_sentences]

In [4]:
hindi_sentences

['<start> नमस्ते <end>',
 '<start> आप कैसे हैं <end>',
 '<start> मैं ठीक हूँ <end>',
 '<start> सुप्रभात <end>',
 '<start> धन्यवाद <end>',
 '<start> मैं तुमसे प्यार करता हूँ <end>',
 '<start> आपका नाम क्या है <end>',
 '<start> मेरा नाम जॉन है <end>']

In [5]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [6]:
eng_tokenizer = Tokenizer(filters="")
eng_tokenizer.fit_on_texts(english_sentences)
eng_sequences = eng_tokenizer.texts_to_sequences(english_sentences)
hin_tokenizer = Tokenizer(filters="")
hin_tokenizer.fit_on_texts(hindi_sentences)
hindi_sequences = hin_tokenizer.texts_to_sequences(hindi_sentences)

In [7]:
print(eng_tokenizer.word_index,"\n" , hin_tokenizer.word_index)

{'you': 1, 'i': 2, 'is': 3, 'name': 4, 'hello': 5, 'how': 6, 'are': 7, 'am': 8, 'fine': 9, 'good': 10, 'morning': 11, 'thank': 12, 'love': 13, 'what': 14, 'your': 15, 'my': 16, 'john': 17} 
 {'<start>': 1, '<end>': 2, 'मैं': 3, 'हूँ': 4, 'नाम': 5, 'है': 6, 'नमस्ते': 7, 'आप': 8, 'कैसे': 9, 'हैं': 10, 'ठीक': 11, 'सुप्रभात': 12, 'धन्यवाद': 13, 'तुमसे': 14, 'प्यार': 15, 'करता': 16, 'आपका': 17, 'क्या': 18, 'मेरा': 19, 'जॉन': 20}


In [8]:
eng_vocab_size = len(eng_tokenizer.word_index) + 1
hin_vocab_size = len(hin_tokenizer.word_index) + 1

In [9]:
max_eng_len = max([len(s) for s in eng_sequences])
max_hin_len = max([len(s) for s in hindi_sequences])

In [10]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [11]:
encoder_input_data = pad_sequences(eng_sequences, maxlen = max_eng_len, padding = "post")
decoder_input_data = pad_sequences(hindi_sequences, maxlen = max_hin_len, padding = "post")

In [12]:
decoder_target_data = np.zeros_like(decoder_input_data)
#decoder_target_data[:,:-1] =  decoder_input_data[:,1:] #first columns of decoder_target_data except last one = all columns of decoder_input_data except first one

In [13]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Embedding, Dense, RepeatVector, TimeDistributed

In [14]:
model = Sequential()

**RepeatVector** transforms:

```
(batch_size, 256)
```

into:

```
(batch_size, max_hin_len, 256)
```

It literally copies the same vector multiple times.

## Visual Example

If `max_hin_len = 4` and encoder output is:

```
[0.2, 0.7, -0.1]
```

`RepeatVector(4)` makes:

```
[
  [0.2, 0.7, -0.1],
  [0.2, 0.7, -0.1],
  [0.2, 0.7, -0.1],
  [0.2, 0.7, -0.1]
]
```

> It does **NOT** learn anything. It just **copies**.\

In [15]:
embedding_dim = 128
model.add(Embedding(eng_vocab_size,                            #Encoder
                    embedding_dim,
                    input_shape=(max_eng_len,)))

model.add(LSTM(256))
model.add(RepeatVector(max_hin_len))                              #RepeatVector
model.add(LSTM(256, return_sequences=True))
model.add(TimeDistributed(Dense(hin_vocab_size, activation="softmax")))

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 4, 128)         │         2,304 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 256)            │       394,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat_vector (RepeatVector)    │ (None, 7, 256)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 7, 256)         │       525,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed                │ (None, 7, 21)          │         5,397 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 927,253 (3.54 MB)

 Trainable params: 927,253 (3.54 MB)

 Non-trainable params: 0 (0.00 B)

In [16]:
model.fit(
    encoder_input_data,
    decoder_input_data,
    batch_size=2,
    epochs=65,
    verbose=1
)

Epoch 1/65
4/4 ━━━━━━━━━━━━━━━━━━━━ 4s 44ms/step - accuracy: 0.1762 - loss: 3.0309
Epoch 2/65
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.3548 - loss: 2.9070
Epoch 3/65
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.3429 - loss: 2.4136
Epoch 4/65
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.3286 - loss: 2.2741
Epoch 5/65
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 0.3452 - loss: 1.9995
Epoch 6/65
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.3643 - loss: 1.9866
Epoch 7/65
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step - accuracy: 0.2405 - loss: 2.2919
Epoch 8/65
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step - accuracy: 0.3167 - loss: 2.0064
Epoch 9/65
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step - accuracy: 0.2762 - loss: 2.1238
Epoch 10/65
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step - accuracy: 0.2690 - loss: 2.0859
Epoch 11/65
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step - accuracy: 0.3119 - loss: 1.9424
Epoch 12/65
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step - accuracy: 0.2857 - loss: 1.9427
E

In [17]:
reverse_hin_word_index = dict(
    (i, word) for word, i in hin_tokenizer.word_index.items()
)

def translate(sentence):

    seq = eng_tokenizer.texts_to_sequences([sentence])
    seq = pad_sequences(seq, maxlen=max_eng_len, padding='post')

    prediction = model.predict(seq)
    prediction = np.argmax(prediction[0], axis=1)

    words = []
    for idx in prediction:
        if idx == 0:
            continue
        word = reverse_hin_word_index.get(idx, "")
        words.append(word)

    return " ".join(words)

In [28]:
translate("bye")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step


'<start> नमस्ते <end>'